# UniMol v2 — PXR pEC50 Fine-tuning (v2_e3_crhi_84m, 8-Fold, Kaggle)

**Training data**: `train_unimol_e3_ct2_crhi_semi.csv` (NO sc_inactives_300)
- clean_train2 (3,743) + crude_nv_hi (244) + semi_nv (55) = **4,042 compounds**
- HA ≥ 5.5: **467** compounds (+97, +26% vs train_v2_inactive300)
- HA ≥ 6.0: **105** compounds (+41, +64% vs train_v2_inactive300)

**Why NOT sc_inactives_300**: Controlled ablation showed sc_inactives hurt UniMol overall
(+0.0034 RAE). Potent>5.5 bin degrades +0.44 RAE. Chemprop benefits because it uses
sample weights; UniMol treats all rows equally so 300 identical pEC50=2.0 rows distort
the high-end embedding. sc_inactives correction is handled by Chemprop v4_4 in the blend.

**Improvements over v2_semi_crhi (RAE=0.5757)**:
1. **3x oversample pEC50 ≥ 5.5** — 467 → 1401 high-potency rows, mirrors Chemprop v4_4
2. **60 epochs + early_stop=15** (was 40/10)
3. **Raw predictions are primary** — calibration hurts on unblinded test (0.5757→0.6695)
4. Model: UniMol v2 **84m** (164m OOM on T4 14.56 GB)

**Pipeline**
1. Load `train_unimol_e3_ct2_crhi_semi.csv` (4,042 rows)
2. Generate 3D conformers (RDKit ETKDG v3 + MMFF94)
3. 3× oversample pEC50 ≥ 5.5 → 5,010 effective training rows
4. Fine-tune UniMol v2 (84m) — 8-fold scaffold CV, 60 epochs
5. OOF per-bin RAE evaluation (calibration diagnostic only)
6. Test predictions → `predictions_unimol_v2_e3_crhi_84m_8fold_raw.csv`


## Cell 1 — GPU Check

In [ ]:
import subprocess, sys
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
else:
    raise SystemExit('No GPU detected — enable GPU accelerator in Kaggle settings.')

import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## Cell 2 — Install Dependencies

In [ ]:
!pip install -q unimol-tools huggingface_hub

from unimol_tools import MolTrain, MolPredict
from rdkit import Chem
print('unimol-tools + rdkit ready')

## Cell 3 — Configuration

In [ ]:
import os, glob, warnings, time
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTORCH_ALLOC_CONF']      = 'expandable_segments:True'  # reduces OOM from fragmentation

# ── Column names ──────────────────────────────────────────────────────────────
SMILES_COL = 'SMILES'
TARGET_COL = 'pEC50'
NAME_COL   = 'Molecule Name'

# ── Locate input files (auto-detect Kaggle dataset path) ─────────────────────
def find_file(filename):
    for root, dirs, files in os.walk('/kaggle/input'):
        if filename in files:
            return os.path.join(root, filename)
    raise FileNotFoundError(f'{filename} not found under /kaggle/input')

# Training set: clean_train2 + crude_nv_hi (244) + semi_nv (55) — NO sc_inactives_300
# sc_inactives hurt UniMol (+0.44 Potent RAE) — use Chemprop v4_4 for inactive correction
# 467 HA>=5.5, 105 HA>=6.0, 844 inactives<3.5 — all real assay measurements
TRAIN_CSV = find_file('train_unimol_e3_ct2_crhi_semi.csv')
TEST_CSV  = find_file('test.csv')
print(f'Train : {TRAIN_CSV}')
print(f'Test  : {TEST_CSV}')

# ── UniMol hyperparameters ────────────────────────────────────────────────────
UNIMOL_MODEL      = 'unimolv2'
UNIMOL_SIZE       = '84m'      # 164m OOM on T4 (14.56 GB); 84m fits with batch=4
                                   # If OOM persists with batch=2, change to '84m' + batch=4
UNIMOL_EPOCHS     = 60
UNIMOL_BATCH_SIZE = 4
UNIMOL_LR         = 1e-4
UNIMOL_EARLY_STOP = 15
UNIMOL_KFOLD      = 8          # 8-fold scaffold CV
UNIMOL_SPLIT      = 'scaffold'

# ── Conformer generation ──────────────────────────────────────────────────────
CONF_SEED         = 42
CONF_MAX_ATTEMPTS = 5
REMOVE_HS         = False      # keep H for UniMol

# ── Output paths ─────────────────────────────────────────────────────────────
UNIMOL_SAVE_DIR = '/kaggle/working/unimol_pxr_v2_e3_crhi_84m_8fold'
OUTPUT_CSV      = '/kaggle/working/predictions_unimol_v2_e3_crhi_84m_8fold.csv'
OUTPUT_RAW_CSV  = '/kaggle/working/predictions_unimol_v2_e3_crhi_84m_8fold_raw.csv'
OOF_CSV         = '/kaggle/working/oof_unimol_v2_e3_crhi_84m_8fold.csv'

GLOBAL_SEED = 42
print(f'\nUniMol {UNIMOL_MODEL} ({UNIMOL_SIZE}) — {UNIMOL_KFOLD}-fold scaffold CV')
print(f'Training data : train_unimol_e3_ct2_crhi_semi.csv (4042 rows)')
print(f'Epochs: {UNIMOL_EPOCHS} (early_stop={UNIMOL_EARLY_STOP})  Batch: {UNIMOL_BATCH_SIZE}  LR: {UNIMOL_LR}')


## Cell 4 — Load & Validate Data

In [ ]:
def load_and_validate(train_csv, test_csv):
    train_df = pd.read_csv(train_csv)
    test_df  = pd.read_csv(test_csv)
    train_df.columns = [c.strip() for c in train_df.columns]
    test_df.columns  = [c.strip() for c in test_df.columns]
    for col, df, fname in [(SMILES_COL, train_df, train_csv),
                            (TARGET_COL, train_df, train_csv),
                            (SMILES_COL, test_df,  test_csv)]:
        if col not in df.columns:
            match = next((c for c in df.columns if c.lower() == col.lower()), None)
            if match:
                df.rename(columns={match: col}, inplace=True)
            else:
                raise ValueError(f"Column '{col}' not found in {fname}. "
                                 f"Available: {list(df.columns)}")
    before = len(train_df)
    train_df = train_df.dropna(subset=[SMILES_COL, TARGET_COL]).reset_index(drop=True)
    test_df  = test_df.dropna(subset=[SMILES_COL]).reset_index(drop=True)
    if len(train_df) < before:
        print(f'  Dropped {before - len(train_df)} rows with missing values')
    return train_df, test_df

train_df, test_df = load_and_validate(TRAIN_CSV, TEST_CSV)
y_train = train_df[TARGET_COL].values.astype(np.float32)

n_ha55  = (y_train >= 5.5).sum()
n_ha60  = (y_train >= 6.0).sum()
n_inact = (y_train < 3.5).sum()
print(f'Train : N={len(train_df)}  '
      f'pEC50=[{y_train.min():.2f}, {y_train.max():.2f}]  '
      f'mean={y_train.mean():.3f}')
print(f'        HA>=5.5: {n_ha55}  HA>=6.0: {n_ha60}  Inact<3.5: {n_inact}')
print(f'Test  : N={len(test_df)}')


## Cell 5 — Generate 3D Conformers with RDKit ETKDG

Conformers are generated **on this hardware** with a fixed seed to ensure reproducibility within this run. ETKDG v3 + MMFF94 minimisation gives geometry quality comparable to the UniMol pretraining data.

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem

def generate_conformers(smiles_list, seed=42, max_attempts=5, remove_hs=False):
    """
    Generate one 3D conformer per molecule using RDKit ETKDG v3 + MMFF94.
    Returns (atoms_list, coords_list, failed_indices).
    - atoms_list  : list of element symbol lists
    - coords_list : list of (N, 3) float64 arrays
    - failed      : indices where conformer generation failed
    """
    atoms_list, coords_list, failed = [], [], []
    params = AllChem.ETKDGv3()
    params.randomSeed        = seed
    params.numThreads        = 0       # use all available cores
    params.enforceChirality  = True
    params.useSmallRingTorsions = True

    for i, smi in enumerate(smiles_list):
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            print(f'  WARNING: mol {i} invalid SMILES — skipping')
            atoms_list.append(None)
            coords_list.append(None)
            failed.append(i)
            continue

        mol_h = Chem.AddHs(mol)
        ok = -1
        for attempt in range(max_attempts):
            p = AllChem.ETKDGv3()
            p.randomSeed = seed + attempt
            p.numThreads = 0
            p.enforceChirality = True
            p.useSmallRingTorsions = True
            ok = AllChem.EmbedMolecule(mol_h, p)
            if ok == 0:
                break

        if ok != 0:
            # Last resort: distance geometry without constraints
            ok = AllChem.EmbedMolecule(mol_h, AllChem.ETKDG())

        if ok != 0:
            print(f'  WARNING: conformer failed for mol {i} ({smi[:40]}) — using None')
            atoms_list.append(None)
            coords_list.append(None)
            failed.append(i)
            continue

        # MMFF94 minimisation
        try:
            AllChem.MMFFOptimizeMolecule(mol_h, maxIters=2000)
        except Exception:
            pass  # keep unminimised geometry

        if remove_hs:
            mol_h = Chem.RemoveHs(mol_h)

        atoms  = [a.GetSymbol() for a in mol_h.GetAtoms()]
        coords = mol_h.GetConformer().GetPositions().astype(np.float64)
        atoms_list.append(atoms)
        coords_list.append(coords)

    return atoms_list, coords_list, failed


print('Generating train conformers...')
t0 = time.time()
train_atoms, train_coords, train_failed = generate_conformers(
    train_df[SMILES_COL].tolist(), seed=CONF_SEED,
    max_attempts=CONF_MAX_ATTEMPTS, remove_hs=REMOVE_HS)
print(f'  Done in {(time.time()-t0)/60:.1f} min  |  failed: {len(train_failed)}')
if train_atoms[0] is not None:
    print(f'  Example: {len(train_atoms[0])} atoms, coords shape {train_coords[0].shape}')

print('Generating test conformers...')
t0 = time.time()
test_atoms, test_coords, test_failed = generate_conformers(
    test_df[SMILES_COL].tolist(), seed=CONF_SEED,
    max_attempts=CONF_MAX_ATTEMPTS, remove_hs=REMOVE_HS)
print(f'  Done in {(time.time()-t0)/60:.1f} min  |  failed: {len(test_failed)}')

print(f'\nConformers ready — train: {len(train_atoms)}  test: {len(test_atoms)}')

## Cell 6 — 8-Fold UniMol Fine-tuning

Filters conformer failures, then fine-tunes UniMol v2 with 8-fold scaffold CV.


In [ ]:
from unimol_tools import MolTrain

os.makedirs(UNIMOL_SAVE_DIR, exist_ok=True)

# ── Filter out molecules where conformer generation failed ────────────────
valid_mask   = [a is not None for a in train_atoms]
n_failed     = sum(1 for v in valid_mask if not v)
n_valid      = sum(valid_mask)

if n_failed > 0:
    print(f'WARNING: {n_failed} conformers failed — filtering to {n_valid} valid.')
    filt_smiles  = [s for s, v in zip(train_df[SMILES_COL].tolist(), valid_mask) if v]
    filt_atoms   = [a for a, v in zip(train_atoms,  valid_mask) if v]
    filt_coords  = [c for c, v in zip(train_coords, valid_mask) if v]
    filt_targets = [t for t, v in zip(train_df[TARGET_COL].tolist(), valid_mask) if v]
    failed_smi   = [s for s, v in zip(train_df[SMILES_COL].tolist(), valid_mask) if not v]
    print(f'  Failed SMILES (first 5): {failed_smi[:5]}')
else:
    print(f'All {n_valid} conformers generated successfully.')
    filt_smiles  = train_df[SMILES_COL].tolist()
    filt_atoms   = train_atoms
    filt_coords  = train_coords
    filt_targets = train_df[TARGET_COL].tolist()

# ── Oversample pEC50 >= 5.5 at 3x (same strategy as Chemprop v4_4) ──────────
filt_y_arr = np.array(filt_targets, dtype=np.float32)
ha55_idx   = np.where(filt_y_arr >= 5.5)[0]
aug_smiles  = list(filt_smiles)
aug_atoms   = list(filt_atoms)
aug_coords  = list(filt_coords)
aug_targets = list(filt_targets)
for _ in range(2):  # add 2 copies → 3x total
    for i in ha55_idx:
        aug_smiles.append(filt_smiles[i])
        aug_atoms.append(filt_atoms[i])
        aug_coords.append(filt_coords[i])
        aug_targets.append(filt_targets[i])
n_aug = len(ha55_idx) * 2
print(f'Oversampling : {len(ha55_idx)} compounds with pEC50>=5.5 added 2x extra '
      f'-> {len(aug_smiles)} total training rows (was {len(filt_smiles)})')

train_data = {
    'SMILES'      : aug_smiles,
    'atoms'       : aug_atoms,
    'coordinates' : aug_coords,
    TARGET_COL    : aug_targets,
}

print('=' * 65)
print(f'UniMol {UNIMOL_MODEL} ({UNIMOL_SIZE}) — {UNIMOL_KFOLD}-Fold Fine-tuning')
print(f'Train samples : {len(aug_smiles)}  (of {len(train_df)} raw, {len(filt_smiles)} valid, +{n_aug} oversampled)')
print(f'Epochs        : up to {UNIMOL_EPOCHS}  early_stop={UNIMOL_EARLY_STOP}')
print(f'Batch / LR    : {UNIMOL_BATCH_SIZE} / {UNIMOL_LR}')
print('=' * 65)

t0 = time.time()
clf = MolTrain(
    task           = 'regression',
    data_type      = 'molecule',
    epochs         = UNIMOL_EPOCHS,
    learning_rate  = UNIMOL_LR,
    batch_size     = UNIMOL_BATCH_SIZE,
    early_stopping = UNIMOL_EARLY_STOP,
    metrics        = 'mae',
    split          = UNIMOL_SPLIT,
    kfold          = UNIMOL_KFOLD,
    save_path      = UNIMOL_SAVE_DIR,
    remove_hs      = REMOVE_HS,
    smiles_col     = 'SMILES',
    target_cols    = TARGET_COL,
    model_name     = UNIMOL_MODEL,
    model_size     = UNIMOL_SIZE,
    use_cuda       = True,
    seed           = GLOBAL_SEED,
)
fit_result = clf.fit(data=train_data)
t_elapsed  = (time.time() - t0) / 60
print(f'\nFine-tuning complete in {t_elapsed:.1f} min')
model_files = sorted(glob.glob(os.path.join(UNIMOL_SAVE_DIR, 'model_*.pth')))
print(f'Saved models: {[os.path.basename(m) for m in model_files]}')


## Cell 7 — OOF Evaluation (+ Calibration for Diagnostics Only)

OOF uses **oversampled** training SMILES (`aug_smiles`).  
**Raw predictions are primary** — calibration hurt RAE 0.5757→0.6695 on v2_semi_crhi.  
Use `predictions_unimol_v2_e3_crhi_84m_8fold_raw.csv` for blending.


In [ ]:
import shutil, tempfile, re
from unimol_tools import MolPredict
from sklearn.isotonic import IsotonicRegression
from rdkit.Chem.Scaffolds import MurckoScaffold
from collections import defaultdict

def scaffold_kfold_indices(smiles_list, n_folds, seed=42):
    scaffold_to_idx = defaultdict(list)
    for i, smi in enumerate(smiles_list):
        try:
            mol  = Chem.MolFromSmiles(smi)
            scaf = MurckoScaffold.MurckoScaffoldSmiles(
                mol=mol, includeChirality=False) if mol else smi
        except:
            scaf = smi
        scaffold_to_idx[scaf].append(i)
    groups     = sorted(scaffold_to_idx.values(), key=len, reverse=True)
    folds      = [[] for _ in range(n_folds)]
    fold_sizes = [0] * n_folds
    for group in groups:
        smallest = min(range(n_folds), key=lambda x: fold_sizes[x])
        folds[smallest].extend(group)
        fold_sizes[smallest] += len(group)
    return [np.array(f) for f in folds]

# Use filt_smiles/filt_targets — the molecules actually used in training
filt_y           = np.array(filt_targets, dtype=np.float32)
fold_val_indices = scaffold_kfold_indices(filt_smiles, n_folds=UNIMOL_KFOLD, seed=GLOBAL_SEED)
y_oof            = np.full(len(filt_y), np.nan)
config_src       = os.path.join(UNIMOL_SAVE_DIR, 'config.yaml')
support_files    = ['target_scaler.ss', 'metric.result']

print('Running per-fold OOF predictions...')
for fold_i, (model_path, val_idx) in enumerate(zip(model_files, fold_val_indices)):
    tmp_dir = tempfile.mkdtemp(prefix=f'fold{fold_i}_')
    try:
        shutil.copy(model_path, os.path.join(tmp_dir, 'model_0.pth'))
        for sf in support_files:
            src = os.path.join(UNIMOL_SAVE_DIR, sf)
            if os.path.exists(src):
                shutil.copy(src, os.path.join(tmp_dir, sf))
        with open(config_src) as f:
            cfg = f.read()
        for key in ['kfold', 'fold_num', 'n_model', 'num_fold']:
            cfg = re.sub(rf'{key}\s*:\s*\d+', f'{key}: 1', cfg)
        with open(os.path.join(tmp_dir, 'config.yaml'), 'w') as f:
            f.write(cfg)
        val_smi    = [filt_smiles[i] for i in val_idx]
        val_df_tmp = pd.DataFrame({'SMILES': val_smi, TARGET_COL: 0.0})
        pred_fold  = MolPredict(load_model=tmp_dir)
        fold_preds = np.array(pred_fold.predict(data=val_df_tmp)).flatten()
        y_oof[val_idx] = fold_preds
        fold_mae = float(np.mean(np.abs(filt_y[val_idx] - fold_preds)))
        fold_sp  = float(spearmanr(filt_y[val_idx], fold_preds).statistic)
        print(f'  fold {fold_i}: N={len(val_idx)}  MAE={fold_mae:.4f}  Spearman={fold_sp:.4f}')
    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)

missing = np.isnan(y_oof)
if missing.any():
    print(f'  WARNING: {missing.sum()} OOF missing — filling with mean')
    y_oof[missing] = filt_y.mean()

def rae(y, yh): return float(np.sum(np.abs(y-yh)) / np.sum(np.abs(y-y.mean())))
oof_rae = rae(filt_y, y_oof)
oof_mae = float(np.mean(np.abs(filt_y - y_oof)))
oof_sp  = float(spearmanr(filt_y, y_oof).statistic)
print(f'\nOOF  RAE={oof_rae:.4f}  MAE={oof_mae:.4f}  Spearman={oof_sp:.4f}')
print(f'OOF pred range : [{y_oof.min():.3f}, {y_oof.max():.3f}]')
print(f'True     range : [{filt_y.min():.3f}, {filt_y.max():.3f}]')
print(f'Compression    : {y_oof.std()/filt_y.std():.3f}  (1.0 = none)')

# ── Per-bin OOF RAE breakdown ────────────────────────────────────────────────
def rae_bin(y, yh, mask):
    if mask.sum() == 0: return float('nan')
    return float(np.sum(np.abs(y[mask]-yh[mask])) / np.sum(np.abs(y[mask]-y[mask].mean())))

print('\nOOF per-bin RAE:')
for label, mask in [('Low  <4.0',  filt_y < 4.0),
                     ('Mid  4-5',   (filt_y>=4.0)&(filt_y<5.0)),
                     ('High 5-5.5', (filt_y>=5.0)&(filt_y<5.5)),
                     ('Potent >5.5', filt_y>=5.5)]:
    print(f'  {label:12s} N={mask.sum():4d}  RAE={rae_bin(filt_y, y_oof, mask):.4f}  '
          f'bias={float(np.mean(y_oof[mask]-filt_y[mask])):+.4f}')

# ── Isotonic calibration (kept for diagnostics — use RAW for blending) ────────
# NOTE: calibration consistently HURTS on unblinded test (RAE 0.5757->0.6695
#       for v2_semi_crhi). Raw predictions are better. Calibration is fit here
#       only for diagnostic comparison; test set is saved as RAW primary.
iso_reg   = IsotonicRegression(out_of_bounds='clip', increasing=True)
iso_reg.fit(y_oof, filt_y)
y_oof_cal = iso_reg.predict(y_oof)
cal_mae   = float(np.mean(np.abs(filt_y - y_oof_cal)))
cal_sp    = float(spearmanr(filt_y, y_oof_cal).statistic)
print(f'\nPost-calibration  MAE={cal_mae:.4f}  Spearman={cal_sp:.4f}')
print('  (Calibrated predictions are for reference only — do NOT use for blending)')


## Cell 8 — Test Set Predictions

Guard handles the edge case where `MolPredict` silently drops failed conformers,
returning fewer predictions than test compounds.


In [ ]:
from unimol_tools import MolPredict

n_test    = len(test_df)
test_data = {
    'SMILES'      : test_df[SMILES_COL].tolist(),
    'atoms'       : test_atoms,
    'coordinates' : test_coords,
    TARGET_COL    : [0.0] * n_test,
}

print(f'Predicting {n_test} test compounds from {UNIMOL_SAVE_DIR}...')
t0 = time.time()
predictor  = MolPredict(load_model=UNIMOL_SAVE_DIR)
raw_preds  = np.array(predictor.predict(data=test_data)).flatten()

# ── Guard: pad if MolPredict dropped failed conformers ───────────────────────
if len(raw_preds) < n_test:
    print(f'  WARNING: got {len(raw_preds)} preds for {n_test} test compounds — padding with mean')
    test_preds_raw = np.full(n_test, float(np.nanmean(raw_preds)))
    valid_idx = [i for i, a in enumerate(test_atoms) if a is not None]
    for k, i in enumerate(valid_idx[:len(raw_preds)]):
        test_preds_raw[i] = raw_preds[k]
else:
    test_preds_raw = raw_preds

test_preds_cal = iso_reg.predict(test_preds_raw)
print(f'Done in {(time.time()-t0)/60:.1f} min')
print(f'Raw : [{test_preds_raw.min():.3f}, {test_preds_raw.max():.3f}]  '
      f'std={test_preds_raw.std():.3f}')
print(f'Cal : [{test_preds_cal.min():.3f}, {test_preds_cal.max():.3f}]  '
      f'std={test_preds_cal.std():.3f}')
print(f'Train ref : [{filt_y.min():.3f}, {filt_y.max():.3f}]  '
      f'std={filt_y.std():.3f}')


## Cell 9 — Save Outputs

Saves three files to `/kaggle/working/`:
- `predictions_unimol_v2_e3_crhi_84m_8fold_raw.csv` — raw predictions (for blending)
- `predictions_unimol_v2_e3_crhi_84m_8fold_cal.csv` — isotonic-calibrated predictions
- `oof_unimol_v2_e3_crhi_84m_8fold.csv` — OOF predictions (for blend weight optimisation)


In [ ]:
name_col = next(
    (c for c in test_df.columns
     if c.lower().replace(' ','_') in ('molecule_name', 'id')), None)

def save_preds(preds, path, label):
    sub = pd.DataFrame()
    if name_col:
        sub['Molecule Name'] = test_df[name_col].values
    else:
        sub['Molecule Name'] = [f'compound_{i+1}' for i in range(len(test_df))]
    sub['SMILES'] = test_df[SMILES_COL].values
    sub['pEC50']  = preds
    sub.to_csv(path, index=False)
    print(f'{label}: {path}')
    print(f'  range=[{preds.min():.3f}, {preds.max():.3f}]  '
          f'mean={preds.mean():.3f}  N={len(sub)}')
    return sub

# ── Test predictions ─────────────────────────────────────────────────────────
# PRIMARY: raw predictions (use these for blending — calibration hurts on test)
save_preds(test_preds_raw, OUTPUT_RAW_CSV, 'RAW (PRIMARY — use for blending)')
# DIAGNOSTIC: calibrated predictions (for reference comparison only)
save_preds(test_preds_cal, OUTPUT_CSV,     'Calibrated (DIAGNOSTIC only)')

# ── OOF predictions (for blend weight optimisation) ──────────────────────────
oof_out = pd.DataFrame({
    'SMILES'      : filt_smiles,
    'pEC50_true'  : filt_targets,
    'pEC50_pred'  : y_oof,
    'pEC50_cal'   : y_oof_cal,
})
oof_out.to_csv(OOF_CSV, index=False)
print(f'OOF: {OOF_CSV}  N={len(oof_out)}')

print(f'\n{"+"*60}')
print(f'OOF  RAE={oof_rae:.4f}  MAE={oof_mae:.4f}  Spearman={oof_sp:.4f}')
print(f'Cal  MAE={cal_mae:.4f}  Spearman={cal_sp:.4f}')
print(f'Files saved to /kaggle/working/')
print(f'  {OUTPUT_RAW_CSV}')
print(f'  {OUTPUT_CSV}')
print(f'  {OOF_CSV}')
print(f'{"+"*60}')
